In [ ]:
# =============================================================================
# CÉLULA 0: PARÂMETROS GLOBAIS + CONFIGURAÇÃO DE LOGGING
# =============================================================================
EMAIL_REMETENTE = "ab11.94958191@gmail.com"
SENHA_APP = ""

PARAMS_BAIXA_VOL = {
    'kelly_frac': 0.30,
    'wyckoff_threshold': 0.75,
    'gap_max_pct': 0.03,
    'custos_pct': 0.003,
    'exigir_volume_anormal': False
}

PARAMS_ALTA_VOL = {
    'kelly_frac': 0.15,
    'wyckoff_threshold': 0.85,
    'gap_max_pct': 0.015,
    'custos_pct': 0.006,
    'exigir_volume_anormal': True
}

PARAMS_ATIVOS = PARAMS_BAIXA_VOL.copy()

MAX_SETUPS_POR_DIA = 5
MAX_PERDAS_CONSECUTIVAS = 3
DRAWDOWN_MAX_DIARIO = 0.02
MAX_DIAS_LOG = 30

HABILITAR_LOGGING = True
ARQUIVO_LOG = "trading_log_v47.json"
ARQUIVO_LOG_DETALHADO = "execucao_detalhada_v47.log"

CAPITAL_TOTAL = 100000.0
WIN_RATE_ESTIMADO = 0.40
PAYOFF_ESTIMADO = 3.0

SETORES_BLOQUEADOS = ['AEREA']
TICKERS_BLOQUEADOS = ['GFSA3.SA', 'ONCO3.SA', 'PMAM3.SA', 'AZTE3.SA', 'RAIZ4.SA',
                      'BHIA3.SA', 'CASH3.SA', 'LJQQ3.SA', 'RCSL4.SA', 'HBOR3.SA']

FALLBACK_TICKERS = [
    'PETR4.SA', 'VALE3.SA', 'ITUB4.SA', 'BBDC4.SA', 'BBAS3.SA', 'ABEV3.SA',
    'WEGE3.SA', 'RADL3.SA', 'SUZB3.SA', 'GGBR4.SA', 'MGLU3.SA', 'VVAR3.SA',
    'RENT3.SA', 'RAIL3.SA', 'CCRO3.SA', 'ELET3.SA', 'CPFE3.SA', 'SBSP3.SA',
    'SANB11.SA', 'B3SA3.SA', 'JBSS3.SA', 'BRFS3.SA', 'KLBN11.SA', 'EQTL3.SA'
]

# ✅ Logging de performance
LOG_PERFORMANCE = True
LOG_FILTROS_DETALHADO = True


In [ ]:
# =============================================================================
# CÉLULA 1: INSTALAÇÃO, IMPORTAÇÕES E SISTEMA DE LOGGING
# =============================================================================
!pip install yfinance pandas-ta --quiet

import yfinance as yf
import pandas as pd
import numpy as np
import pandas_ta as ta
import requests
from bs4 import BeautifulSoup
import smtplib
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText
from datetime import datetime, timedelta
import time, warnings, json, os, sys
from typing import Optional, Tuple, Dict, List
import traceback

warnings.filterwarnings("ignore")

# ✅ SISTEMA DE LOGGING COM TIMESTAMP E DURAÇÃO
class Logger:
    def __init__(self, arquivo_log: str, arquivo_detalhado: str = None):
        self.arquivo_log = arquivo_log
        self.arquivo_detalhado = arquivo_detalhado
        self.inicio_geral = time.time()
        self.timings = {}
        self.contadores = {}
        
    def log(self, mensagem: str, nivel: str = "INFO", ticker: str = None):
        timestamp = datetime.now().strftime("%H:%M:%S")
        log_msg = f"[{timestamp}] [{nivel}] {mensagem}"
        if ticker:
            log_msg += f" | {ticker}"
        print(log_msg)
        if self.arquivo_detalhado and LOG_PERFORMANCE:
            with open(self.arquivo_detalhado, 'a', encoding='utf-8') as f:
                f.write(log_msg + "\n")
                
    def iniciar_etapa(self, nome_etapa: str):
        self.timings[nome_etapa] = {'inicio': time.time()}
        self.log(f"🚀 Iniciando: {nome_etapa}", "ETAPA")
        
    def concluir_etapa(self, nome_etapa: str, detalhes: dict = None):
        if nome_etapa in self.timings:
            duracao = time.time() - self.timings[nome_etapa]['inicio']
            self.timings[nome_etapa]['duracao'] = duracao
            msg = f"✅ Concluído: {nome_etapa} ({duracao:.2f}s)"
            if detalhes:
                detalhes_str = " | ".join(f"{k}: {v}" for k, v in detalhes.items())
                msg += f" | {detalhes_str}"
            self.log(msg, "ETAPA")
            
    def incrementar(self, contador: str, valor: int = 1):
        self.contadores[contador] = self.contadores.get(contador, 0) + valor
        
    def resumo_final(self):
        duracao_total = time.time() - self.inicio_geral
        self.log("\n" + "="*60, "RESUMO")
        self.log(f"⏱️ Tempo total: {duracao_total:.2f}s", "RESUMO")
        self.log("\n📊 Duração por etapa:", "RESUMO")
        for etapa, dados in self.timings.items():
            if 'duracao' in dados:
                pct = (dados['duracao'] / duracao_total) * 100
                self.log(f"   • {etapa}: {dados['duracao']:.2f}s ({pct:.1f}%)", "RESUMO")
        if self.contadores:
            self.log("\n🔢 Contadores:", "RESUMO")
            for cont, val in self.contadores.items():
                self.log(f"   • {cont}: {val}", "RESUMO")
        self.log("="*60 + "\n", "RESUMO")
        
        # Salvar resumo em JSON para análise posterior
        resumo = {
            'timestamp': datetime.now().isoformat(),
            'duracao_total_segundos': duracao_total,
            'timings': self.timings,
            'contadores': self.contadores
        }
        with open('resumo_execucao.json', 'w', encoding='utf-8') as f:
            json.dump(resumo, f, indent=2, ensure_ascii=False)

# Inicializar logger
logger = Logger(ARQUIVO_LOG, ARQUIVO_LOG_DETALHADO)

def log_evento(tipo: str, ticker: str, dados: dict, arquivo: str = ARQUIVO_LOG, max_dias: int = MAX_DIAS_LOG):
    if not HABILITAR_LOGGING:
        return
    registro = {'timestamp': datetime.now().isoformat(), 'tipo': tipo, 'ticker': ticker, 'dados': dados}
    logs = []
    if os.path.exists(arquivo):
        try:
            with open(arquivo, 'r', encoding='utf-8') as f:
                logs = json.load(f)
        except:
            logs = []
    cutoff = datetime.now() - timedelta(days=max_dias)
    logs = [l for l in logs if datetime.fromisoformat(l['timestamp']) > cutoff]
    logs.append(registro)
    with open(arquivo, 'w', encoding='utf-8') as f:
        json.dump(logs, f, ensure_ascii=False, indent=2)

logger.log("🔧 Sistema de logging inicializado", "INFO")
print("✅ Célula 1 carregada.")


In [ ]:
# =============================================================================
# CÉLULA 2: FUNÇÕES AUXILIARES (COM LOGGING OPCIONAL)
# =============================================================================

def calcular_eficiencia_candle(df: pd.DataFrame) -> pd.Series:
    corpo = abs(df['Close'] - df['Open'])
    sombra_sup = df['High'] - df[['Close', 'Open']].max(axis=1)
    sombra_inf = df[['Close', 'Open']].min(axis=1) - df['Low']
    range_total = df['High'] - df['Low'].replace(0, np.nan)
    eficiencia = pd.Series(index=df.index, dtype=float)
    alta, baixa = df['Close'] > df['Open'], df['Close'] < df['Open']
    eficiencia[alta] = 1 - (sombra_sup[alta] / range_total[alta])
    eficiencia[baixa] = 1 - (sombra_inf[baixa] / range_total[baixa])
    return eficiencia

def detectar_regime(df: pd.DataFrame, janela: int = 20) -> pd.Series:
    df_temp = df.copy()
    df_temp['retorno'] = df_temp['Close'].pct_change()
    df_temp['volatilidade'] = df_temp['retorno'].rolling(janela).std()
    adx = ta.adx(df_temp['High'], df_temp['Low'], df_temp['Close'], length=14)
    df_temp['adx'] = adx['ADX_14']
    df_temp = df_temp.dropna(subset=['volatilidade', 'adx'])
    if df_temp.empty: return pd.Series(index=df.index, dtype=int)
    v_p33, v_p67 = df_temp['volatilidade'].quantile([0.33, 0.67])
    a_p33, a_p67 = df_temp['adx'].quantile([0.33, 0.67])
    def classificar(row):
        v, a = row['volatilidade'], row['adx']
        if v < v_p33 and a < a_p33: return 0
        elif v > v_p67 or a > a_p67: return 2
        else: return 1
    regimes = df_temp.apply(classificar, axis=1)
    regime_series = pd.Series(index=df.index, dtype=int)
    regime_series.loc[regimes.index] = regimes
    regime_series.ffill(inplace=True)
    return regime_series

def detectar_swing_low(df: pd.DataFrame, janela: int = 10, confirmar: bool = True) -> Optional[float]:
    lows, closes = df['Low'].values, df['Close'].values
    swing_lows = []
    for i in range(janela, len(lows)):
        if i - janela >= 0 and lows[i] <= min(lows[i-janela:i]):
            if not confirmar or (i+1 < len(lows) and closes[i+1] > lows[i]):
                swing_lows.append(lows[i])
    return float(swing_lows[-1]) if swing_lows else float(df['Low'].min())

def detectar_swing_high(df: pd.DataFrame, janela: int = 10, confirmar: bool = True) -> Optional[float]:
    highs, closes = df['High'].values, df['Close'].values
    swing_highs = []
    for i in range(janela, len(highs)):
        if i - janela >= 0 and highs[i] >= max(highs[i-janela:i]):
            if not confirmar or (i+1 < len(highs) and closes[i+1] < highs[i]):
                swing_highs.append(highs[i])
    return float(swing_highs[-1]) if swing_highs else float(df['High'].max())

def calcular_lta_pivos(df: pd.DataFrame, janela_pivo: int = 5) -> Optional[float]:
    lows = df['Low'].values
    lows_seguro = np.where(lows <= 0, np.nan, lows)
    log_lows = np.log(lows_seguro)
    fundos = []
    for i in range(janela_pivo, len(log_lows) - janela_pivo):
        if np.isnan(log_lows[i]): continue
        if log_lows[i] == min(log_lows[i-janela_pivo:i+janela_pivo+1]):
            fundos.append((i, log_lows[i]))
    if len(fundos) >= 2:
        (x1, y1), (x2, y2) = fundos[-2], fundos[-1]
        x_atual = len(log_lows) - 1
        incl = (y2 - y1) / (x2 - x1)
        return np.exp(y2 + incl * (x_atual - x2))
    return None

def calcular_ltb_pivos(df: pd.DataFrame, janela_pivo: int = 5) -> Optional[float]:
    highs = df['High'].values
    highs_seguro = np.where(highs <= 0, np.nan, highs)
    log_highs = np.log(highs_seguro)
    topos = []
    for i in range(janela_pivo, len(log_highs) - janela_pivo):
        if np.isnan(log_highs[i]): continue
        if log_highs[i] == max(log_highs[i-janela_pivo:i+janela_pivo+1]):
            topos.append((i, log_highs[i]))
    if len(topos) >= 2 and topos[-2][1] > topos[-1][1]:
        (x1, y1), (x2, y2) = topos[-2], topos[-1]
        x_atual = len(log_highs) - 1
        incl = (y2 - y1) / (x2 - x1)
        return np.exp(y2 + incl * (x_atual - x2))
    return None

def detectar_padrao_altista(df_diario: pd.DataFrame) -> bool:
    if len(df_diario) < 3: return False
    u, p = df_diario.iloc[-1], df_diario.iloc[-2]
    c_u = abs(u['Close'] - u['Open'])
    r_u = u['High'] - u['Low']
    si_u = min(u['Close'], u['Open']) - u['Low']
    ss_u = u['High'] - max(u['Close'], u['Open'])
    if r_u > 0 and si_u >= 2 * c_u and ss_u <= 0.3 * c_u: return True
    if p['Close'] < p['Open'] and u['Close'] > u['Open']:
        if u['Open'] <= p['Close'] and u['Close'] >= p['Open']: return True
        meio = (p['Open'] + p['Close']) / 2
        if u['Open'] <= p['Close'] and u['Close'] >= meio: return True
    return False

def detectar_padrao_baixista(df_diario: pd.DataFrame) -> bool:
    if len(df_diario) < 3: return False
    u, p = df_diario.iloc[-1], df_diario.iloc[-2]
    c_u = abs(u['Close'] - u['Open'])
    r_u = u['High'] - u['Low']
    si_u = min(u['Close'], u['Open']) - u['Low']
    ss_u = u['High'] - max(u['Close'], u['Open'])
    if r_u > 0 and ss_u >= 2 * c_u and si_u <= 0.3 * c_u: return True
    if p['Close'] > p['Open'] and u['Close'] < u['Open']:
        if u['Open'] >= p['Close'] and u['Close'] <= p['Open']: return True
        meio = (p['Open'] + p['Close']) / 2
        if u['Open'] >= p['Close'] and u['Close'] <= meio: return True
    return False

SENTIMENTO_PADRAO = 0.0

def detectar_volume_anormal(df: pd.DataFrame, periodo: int = 20, limiar: float = 1.5) -> bool:
    if len(df) < periodo: return False
    vol_medio = df['Volume'].rolling(periodo).mean().iloc[-1]
    return df['Volume'].iloc[-1] >= vol_medio * limiar if pd.notna(vol_medio) else False

def fractional_kelly(win_rate: float, payoff_ratio: float, frac: float = 0.25) -> float:
    if payoff_ratio <= 0: return 0.0
    kelly = (payoff_ratio * win_rate - (1 - win_rate)) / payoff_ratio
    return max(0.0, min(kelly, 0.25)) * frac

MACRO_REFERENCE = {
    'VALE3.SA': ('GC=F', 0.6), 'PETR4.SA': ('CL=F', 0.8), 'PETR3.SA': ('CL=F', 0.8),
    'CSNA3.SA': ('GC=F', 0.5), 'GGBR4.SA': ('GC=F', 0.5), 'CAML3.SA': ('WEAT', 0.4),
    'JBSS3.SA': ('WEAT', 0.5), 'ABEV3.SA': ('CORN', 0.3), 'RADL3.SA': ('XLP', 0.3),
    'PRIO3.SA': ('CL=F', 0.7),
}

def verificar_alinhamento_macro(ticker: str, direcao: str, cache_macro: dict) -> Tuple[bool, int]:
    if ticker not in MACRO_REFERENCE: return True, 15
    ref, _ = MACRO_REFERENCE[ticker]
    if ref not in cache_macro or cache_macro[ref] is None or len(cache_macro[ref]) < 55: return True, 15
    precos = cache_macro[ref]
    ema50 = pd.Series(precos).ewm(span=50, adjust=False).mean()
    e_at, e_lag = ema50.iloc[-1], ema50.iloc[-5]
    if pd.isna(e_at) or pd.isna(e_lag): return True, 15
    slope_pos = e_at > e_lag
    if direcao == 'COMPRA' and precos[-1] > e_at and slope_pos: return True, 30
    if direcao == 'VENDA' and precos[-1] < e_at and not slope_pos: return True, 30
    return False, 0

def avaliar_qualidade_volume(df_w: pd.DataFrame, direcao: str) -> Tuple[str, int]:
    vol_u = df_w['Volume'].iloc[-1]
    vol_m = df_w['Volume'].rolling(20).mean().iloc[-1]
    ef = df_w['Eficiencia'].iloc[-1] if 'Eficiencia' in df_w.columns else 0.5
    if pd.isna(vol_m) or vol_m == 0: return 'NEUTRO', 10
    if vol_u > vol_m * 1.5:
        if direcao == 'COMPRA' and ef > 0.7: return 'ALTA_CONVICCAO', 25
        if direcao == 'VENDA' and ef > 0.7: return 'ALTA_CONVICCAO', 25
        if ef < 0.5: return 'POSSIVEL_ARMADILHA', 15
    return 'NEUTRO', 10

def detectar_fase_wyckoff_adaptativo(df_w: pd.DataFrame, suporte: float = None, 
                                     resistencia: float = None, atr_period: int = 14) -> Tuple[str, float]:
    if len(df_w) < 30: return 'INDEFINIDO', 0.0
    atr = ta.atr(df_w['High'], df_w['Low'], df_w['Close'], length=atr_period)
    atr_med = atr.rolling(20).mean()
    vol_rel = atr.iloc[-1] / atr_med.iloc[-1] if pd.notna(atr_med.iloc[-1]) else 1.0
    thr_base = PARAMS_ATIVOS.get('wyckoff_threshold', 0.75)
    thr_comp = max(0.65, min(0.90, thr_base + 0.10 * (vol_rel - 1)))
    
    range_semanal = df_w['High'] - df_w['Low']
    r_med = range_semanal.rolling(20).mean().iloc[-1]
    if range_semanal.iloc[-1] >= r_med * thr_comp: return 'INDEFINIDO', 0.0
    
    px = df_w['Close'].iloc[-1]
    if suporte is None: suporte = df_w['Low'].rolling(20).min().iloc[-1]
    if resistencia is None: resistencia = df_w['High'].rolling(20).max().iloc[-1]
    faixa = resistencia - suporte
    if faixa == 0: return 'INDEFINIDO', 0.0
    
    pos_rel = (px - suporte) / faixa
    candles = df_w.iloc[-8:]
    alta, baixa = candles['Close'] > candles['Open'], candles['Close'] < candles['Open']
    vol_alta = candles.loc[alta, 'Volume'].mean() if alta.any() else 0
    vol_baixa = candles.loc[baixa, 'Volume'].mean() if baixa.any() else 0
    if vol_baixa == 0: return 'INDEFINIDO', 0.0
    
    razao = vol_alta / vol_baixa
    conf = min(1.0, (r_med - range_semanal.iloc[-1]) / r_med) if r_med > 0 else 0.5
    if pos_rel < 0.4 and razao > 1.2: return 'ACUMULACAO', conf
    if pos_rel > 0.6 and razao < 0.8: return 'DISTRIBUICAO', conf
    return 'INDEFINIDO', 0.0

def calcular_alvos_fibonacci(df: pd.DataFrame, direcao: str, fib_window: int = 20) -> Dict[str, float]:
    if len(df) < fib_window: return {}
    try:
        df_rec = df.iloc[-fib_window:]
        sl = detectar_swing_low(df_rec, janela=5, confirmar=False)
        sh = detectar_swing_high(df_rec, janela=5, confirmar=False)
        if sl is None or sh is None or sl >= sh: return {}
        amp = np.log(sh) - np.log(sl)
        base = np.log(sh)
        mults = [1.000, 1.618, 2.618, 4.236]
        if direcao == 'COMPRA':
            return {f'{m*100}%': round(np.exp(base + amp * m), 2) for m in mults}
        return {f'{m*100}%': round(np.exp(base - amp * m), 2) for m in mults}
    except: return {}

def calcular_score_confianca(setup: dict, alinhado_macro: bool, qualidade_volume: str, 
                             score_volume: int, fase_wyckoff: str, wyckoff_conf: float = 1.0) -> float:
    score = (30 if alinhado_macro else 0) + score_volume
    if setup['Eficiência'] and setup['Eficiência'] > 0.8: score += 20
    elif setup['Eficiência'] and setup['Eficiência'] > 0.6: score += 10
    if setup['Regime'] == 2: score += 10
    elif setup['Regime'] == 1: score += 5
    if setup['Direcao'] == 'COMPRA' and fase_wyckoff == 'ACUMULACAO': score += int(20 * wyckoff_conf)
    elif setup['Direcao'] == 'VENDA' and fase_wyckoff == 'DISTRIBUICAO': score += int(20 * wyckoff_conf)
    return min(score, 100) / 100.0

def calcular_alvo_recomendado(setup: dict, margem: float = 0.03) -> Tuple[float, str]:
    fibos = setup.get('Alvos Fibonacci', {})
    if not fibos or '161.8%' not in fibos: return setup['Alvo 3:1'], '3:1'
    fib = fibos['161.8%']
    if setup['Direcao'] == 'COMPRA' and fib <= setup['Resistência'] * (1 + margem): return fib, 'Fib 161.8%'
    if setup['Direcao'] == 'VENDA' and fib >= setup['Suporte'] * (1 - margem): return fib, 'Fib 161.8%'
    return setup['Alvo 3:1'], '3:1'

def calcular_payoff_real(entrada: float, alvo: float, stop: float, custos_pct: float) -> float:
    risco = abs(entrada - stop)
    if risco == 0: return 0.0
    retorno = abs(alvo - entrada) - (entrada * custos_pct * 2)
    return round(max(0, retorno) / risco, 2)

# ✅ CORREÇÃO: Função robusta para detectar regime de volatilidade
def detectar_regime_volatilidade(df: pd.DataFrame, janela: int = 40) -> str:
    """Detecta regime de volatilidade com tratamento robusto de edge cases."""
    try:
        # Converter Series para DataFrame se necessário
        if isinstance(df, pd.Series):
            if df.empty or df.isna().all():
                return 'BAIXA'
            df = pd.DataFrame({'Close': df.values}, index=df.index)
        
        # Validações básicas
        if df is None or df.empty or 'Close' not in df.columns:
            return 'BAIXA'
        if len(df) < janela:
            return 'BAIXA'
        
        # Calcular volatilidade
        ret = df['Close'].pct_change().dropna()
        if ret.empty or len(ret) < 20:
            return 'BAIXA'
            
        vol_at = ret.rolling(20).std().iloc[-1]
        vol_hist = ret.rolling(janela).std().dropna()
        
        if pd.isna(vol_at) or vol_hist.empty:
            return 'BAIXA'
            
        # Classificar regime
        return 'ALTA' if (vol_hist < vol_at).mean() > 0.7 else 'BAIXA'
        
    except Exception as e:
        # Fallback conservador em caso de qualquer erro
        return 'BAIXA'

logger.log("✅ Funções auxiliares carregadas", "INFO")
print("✅ Célula 2 carregada.")


In [ ]:
# =============================================================================
# CÉLULA 3: FUNÇÕES DE ANÁLISE (COM LOGGING DE FILTROS)
# =============================================================================

OTIMIZADO_SWING = {'atr_period': 14, 'atr_mult': 1.8, 'swing_window': 12, 'lta_pivo_window': 6, 'ltb_pivo_window': 6, 'mm200_semanal': True, 'mm200_diaria': True}
OTIMIZADO_POSITION = {'atr_period': 14, 'atr_mult': 2.5, 'swing_window': 24, 'lta_pivo_window': 12, 'ltb_pivo_window': 12, 'mm50_mensal': True}

def analisar_swing_trade(ticker: str, df_w: pd.DataFrame = None, df_d: pd.DataFrame = None) -> Optional[List[dict]]:
    try:
        if df_w is None: return None
        df_w = df_w.copy()
        if not set(['Close','High','Low','Open','Volume']).issubset(df_w.columns):
            df_w.rename(columns={k:v for k,v in {'close':'Close','high':'High','low':'Low','open':'Open','volume':'Volume'}.items() if k in df_w.columns}, inplace=True)
        df_w.sort_index(inplace=True)
        if not isinstance(df_w.index, pd.DatetimeIndex): df_w.index = pd.to_datetime(df_w.index)
        
        df_w['Eficiencia'] = calcular_eficiencia_candle(df_w)
        df_w['Regime'] = detectar_regime(df_w)
        ult = df_w.iloc[-1]
        entrada = float(ult['Close'])
        if pd.isna(entrada) or entrada <= 0: return None
        
        rh = float(df_w['High'].rolling(window=min(52, len(df_w))).max().iloc[-1])
        rl = float(df_w['Low'].rolling(window=min(52, len(df_w))).min().iloc[-1])
        reg = int(ult['Regime']) if not pd.isna(ult['Regime']) else -1
        ef = round(float(ult['Eficiencia']), 2) if not pd.isna(ult['Eficiencia']) else None
        atr = float(ta.atr(df_w['High'], df_w['Low'], df_w['Close'], length=OTIMIZADO_SWING['atr_period']).iloc[-1] or 0.0)
        
        pa = pb = True
        if df_d is not None and not df_d.empty:
            df_loc = df_d.copy()
            if isinstance(df_loc.columns, pd.MultiIndex): df_loc.columns = df_loc.columns.droplevel(1)
            df_loc.columns = ['Close', 'High', 'Low', 'Open', 'Volume']
            pa = detectar_padrao_altista(df_loc)
            pb = detectar_padrao_baixista(df_loc)
            
        sentimento = SENTIMENTO_PADRAO
        vol_an = detectar_volume_anormal(df_w, periodo=20, limiar=1.5)
        setups = []
        
        if pa:
            fib = calcular_alvos_fibonacci(df_w, 'COMPRA')
            st_atr = entrada - (OTIMIZADO_SWING['atr_mult'] * atr) if atr > 0 else None
            st_sw = detectar_swing_low(df_w, janela=OTIMIZADO_SWING['swing_window'])
            st_sw = st_sw if st_sw < entrada else None
            st_lta = calcular_lta_pivos(df_w, janela_pivo=OTIMIZADO_SWING['lta_pivo_window'])
            st_lta = st_lta if st_lta and st_lta < entrada else None
            for met, sl in [('ATR', st_atr), ('Swing Low', st_sw), ('LTA Pivôs', st_lta)]:
                if sl is None or sl <= 0 or sl >= entrada: continue
                r = entrada - sl
                al = entrada + (r * 3)
                if al <= rh * 1.05:
                    setups.append({'Ticker': ticker, 'Modalidade': 'Swing', 'Direcao': 'COMPRA', 'Entrada': round(entrada,2), 'Método Stop': met, 'Stop Loss': round(sl,2), 'Risco (R$)': round(r,2), 'Alvo 3:1': round(al,2), 'Resistência': round(rh,2), 'Suporte': round(rl,2), 'Regime': reg, 'Eficiência': ef, 'Sentimento': sentimento, 'Volume Anormal': vol_an, 'Alvos Fibonacci': fib})
                    
        if pb:
            fib = calcular_alvos_fibonacci(df_w, 'VENDA')
            st_atr = entrada + (OTIMIZADO_SWING['atr_mult'] * atr) if atr > 0 else None
            st_sw = detectar_swing_high(df_w, janela=OTIMIZADO_SWING['swing_window'])
            st_sw = st_sw if st_sw > entrada else None
            st_ltb = calcular_ltb_pivos(df_w, janela_pivo=OTIMIZADO_SWING['ltb_pivo_window'])
            st_ltb = st_ltb if st_ltb and st_ltb > entrada else None
            for met, sl in [('ATR', st_atr), ('Swing High', st_sw), ('LTB Pivôs', st_ltb)]:
                if sl is None or sl <= entrada: continue
                r = sl - entrada
                al = entrada - (r * 3)
                if al >= rl * 0.95:
                    setups.append({'Ticker': ticker, 'Modalidade': 'Swing', 'Direcao': 'VENDA', 'Entrada': round(entrada,2), 'Método Stop': met, 'Stop Loss': round(sl,2), 'Risco (R$)': round(r,2), 'Alvo 3:1': round(al,2), 'Resistência': round(rh,2), 'Suporte': round(rl,2), 'Regime': reg, 'Eficiência': ef, 'Sentimento': sentimento, 'Volume Anormal': vol_an, 'Alvos Fibonacci': fib})
        return setups if setups else None
    except Exception as e:
        if LOG_FILTROS_DETALHADO:
            logger.log(f"Erro em analisar_swing_trade: {str(e)[:100]}", "ERRO", ticker)
        return None

def analisar_position_trade(ticker: str, df_m: pd.DataFrame = None, df_w: pd.DataFrame = None) -> Optional[List[dict]]:
    try:
        if df_m is None: return None
        df_m = df_m.copy()
        if not set(['Close','High','Low','Open','Volume']).issubset(df_m.columns):
            df_m.rename(columns={k:v for k,v in {'close':'Close','high':'High','low':'Low','open':'Open','volume':'Volume'}.items() if k in df_m.columns}, inplace=True)
        df_m.sort_index(inplace=True)
        if not isinstance(df_m.index, pd.DatetimeIndex): df_m.index = pd.to_datetime(df_m.index)
        
        df_m['Eficiencia'] = calcular_eficiencia_candle(df_m)
        df_m['Regime'] = detectar_regime(df_m)
        ult = df_m.iloc[-1]
        entrada = float(ult['Close'])
        if pd.isna(entrada) or entrada <= 0: return None
        
        lb = min(60, len(df_m))
        rh = float(df_m['High'].rolling(window=lb).max().iloc[-1])
        rl = float(df_m['Low'].rolling(window=lb).min().iloc[-1])
        reg = int(ult['Regime']) if not pd.isna(ult['Regime']) else -1
        ef = round(float(ult['Eficiencia']), 2) if not pd.isna(ult['Eficiencia']) else None
        atr = float(ta.atr(df_m['High'], df_m['Low'], df_m['Close'], length=OTIMIZADO_POSITION['atr_period']).iloc[-1] or 0.0)
        
        sentimento = SENTIMENTO_PADRAO
        vol_an = detectar_volume_anormal(df_m, periodo=20, limiar=1.5)
        setups = []
        
        fib_c = calcular_alvos_fibonacci(df_w, 'COMPRA') if df_w is not None else {}
        st_atr = entrada - (OTIMIZADO_POSITION['atr_mult'] * atr) if atr > 0 else None
        st_sw = detectar_swing_low(df_m, janela=OTIMIZADO_POSITION['swing_window'])
        st_sw = st_sw if st_sw and st_sw < entrada else None
        st_lta = calcular_lta_pivos(df_m, janela_pivo=OTIMIZADO_POSITION['lta_pivo_window'])
        st_lta = st_lta if st_lta and st_lta < entrada else None
        for met, sl in [('ATR', st_atr), ('Swing Low', st_sw), ('LTA Pivôs', st_lta)]:
            if sl is None or sl <= 0 or sl >= entrada: continue
            r = entrada - sl
            al = entrada + (r * 3)
            if al <= rh * 1.10:
                setups.append({'Ticker': ticker, 'Modalidade': 'Position', 'Direcao': 'COMPRA', 'Entrada': round(entrada,2), 'Método Stop': met, 'Stop Loss': round(sl,2), 'Risco (R$)': round(r,2), 'Alvo 3:1': round(al,2), 'Resistência': round(rh,2), 'Suporte': round(rl,2), 'Regime': reg, 'Eficiência': ef, 'Sentimento': sentimento, 'Volume Anormal': vol_an, 'Alvos Fibonacci': fib_c})
                
        fib_v = calcular_alvos_fibonacci(df_w, 'VENDA') if df_w is not None else {}
        st_atr = entrada + (OTIMIZADO_POSITION['atr_mult'] * atr) if atr > 0 else None
        st_sw = detectar_swing_high(df_m, janela=OTIMIZADO_POSITION['swing_window'])
        st_sw = st_sw if st_sw and st_sw > entrada else None
        st_ltb = calcular_ltb_pivos(df_m, janela_pivo=OTIMIZADO_POSITION['ltb_pivo_window'])
        st_ltb = st_ltb if st_ltb and st_ltb > entrada else None
        for met, sl in [('ATR', st_atr), ('Swing High', st_sw), ('LTB Pivôs', st_ltb)]:
            if sl is None or sl <= entrada: continue
            r = sl - entrada
            al = entrada - (r * 3)
            if al >= rl * 0.90:
                setups.append({'Ticker': ticker, 'Modalidade': 'Position', 'Direcao': 'VENDA', 'Entrada': round(entrada,2), 'Método Stop': met, 'Stop Loss': round(sl,2), 'Risco (R$)': round(r,2), 'Alvo 3:1': round(al,2), 'Resistência': round(rh,2), 'Suporte': round(rl,2), 'Regime': reg, 'Eficiência': ef, 'Sentimento': sentimento, 'Volume Anormal': vol_an, 'Alvos Fibonacci': fib_v})
        return setups if setups else None
    except Exception as e:
        if LOG_FILTROS_DETALHADO:
            logger.log(f"Erro em analisar_position_trade: {str(e)[:100]}", "ERRO", ticker)
        return None

logger.log("✅ Funções de análise carregadas", "INFO")
print("✅ Célula 3 carregada.")


In [ ]:
# =============================================================================
# CÉLULA 4: EXECUÇÃO PRINCIPAL COM LOGGING DETALHADO
# =============================================================================

try:
    from google.colab import userdata
    if not SENHA_APP: SENHA_APP = userdata.get('GMAIL_APP_PASSWORD')
except: pass

VOLUME_MINIMO_ACAO = 1_000_000
VOLUME_FINANCEIRO_MINIMO = 1_000_000
LIMITE_LIQUIDEZ_FINANCEIRA = 5_000_000
PRECO_MINIMO = 5.00
RISCO_PERCENTUAL_MINIMO = 0.02
RISCO_PERCENTUAL_MAXIMO = 0.20
EXIGIR_CONFLUENCIA = True

def montar_tabela_html(oportunidades: List[dict], titulo: str, regime_vol: str) -> str:
    if not oportunidades: return ""
    corpo = f"<h3>{titulo}</h3><table border='1' cellpadding='4' cellspacing='0' style='border-collapse:collapse;'>"
    corpo += "<tr><th>Ticker</th><th>Dir.</th><th>Entrada</th><th>Stop</th><th>Alvo Rec.</th><th>Método</th><th>Payoff Real</th><th>Vol. Anormal</th><th>Lote</th><th>Score</th></tr>"
    for op in oportunidades:
        vol_an_icon = '✅' if op.get('Volume Anormal') else '❌'
        corpo += f"<tr><td>{op['Ticker']}</td><td>{op['Direcao']}</td><td>R$ {op['Entrada']:.2f}</td><td>R$ {op['Stop Loss']:.2f}</td><td>R$ {op['Alvo Recomendado']:.2f}</td><td>{op['Método Alvo']}</td><td>{op['Payoff Real']}:1</td><td>{vol_an_icon}</td><td>{op['Lote']}</td><td>{op.get('Score', 'N/A')}</td></tr>"
    corpo += f"</table><br><p><small>Custos: {PARAMS_ATIVOS['custos_pct']*100:.1f}% | Regime: {regime_vol}</small></p>"
    return corpo

def enviar_email_ou_exibir(oportunidades: List[dict], modalidade: str, regime_vol: str):
    if not oportunidades:
        logger.log(f"Nenhuma oportunidade de {modalidade} encontrada", "INFO")
        return
    if EMAIL_REMETENTE and SENHA_APP:
        try:
            msg = MIMEMultipart()
            msg['From'] = EMAIL_REMETENTE
            msg['To'] = EMAIL_REMETENTE
            msg['Subject'] = f"🚨 Oportunidades {modalidade} - {datetime.now().strftime('%d/%m/%Y')}"
            msg.attach(MIMEText(montar_tabela_html(oportunidades, "Setups Aprovados", regime_vol), 'html'))
            with smtplib.SMTP_SSL('smtp.gmail.com', 465) as server:
                server.login(EMAIL_REMETENTE, SENHA_APP)
                server.send_message(msg)
            logger.log(f"E-mail ({modalidade}) enviado", "INFO")
        except Exception as e:
            logger.log(f"Falha no e-mail: {str(e)[:100]}", "ERRO")
            log_evento('ERRO_EMAIL', 'SISTEMA', {'erro': str(e)})
    else:
        logger.log(f"E-mail não configurado. Exibindo na tela", "INFO")
    df_op = pd.DataFrame(oportunidades)
    cols = ['Ticker', 'Direcao', 'Entrada', 'Método Stop', 'Stop Loss', 'Risco (R$)', 'Alvo 3:1', 'Alvo Recomendado', 'Payoff Real', 'Resistência', 'Suporte', 'Regime', 'Eficiência', 'Volume Anormal', 'Fase Wyckoff', 'Score', 'Kelly %', 'Lote']
    try:
        from IPython.display import display
        display(df_op[cols].sort_values(['Direcao', 'Ticker']))
    except:
        print(df_op[cols].sort_values(['Direcao', 'Ticker']).to_string())
    csv_name = f"oportunidades_{modalidade.lower()}_{datetime.now().strftime('%Y%m%d')}.csv"
    df_op[cols].to_csv(csv_name, index=False)
    try:
        from google.colab import files
        files.download(csv_name)
    except: logger.log(f"Arquivo '{csv_name}' salvo", "INFO")

def obter_tickers_b3() -> List[str]:
    try:
        url = "https://www.dadosdemercado.com.br/acoes"
        soup = BeautifulSoup(requests.get(url, timeout=10).content, 'html.parser')
        tickers = [row.find_all('td')[0].text.strip() for row in soup.select('table tbody tr') if row.find_all('td') and not row.find_all('td')[0].text.strip().startswith('#')]
        return tickers if tickers else FALLBACK_TICKERS.copy()
    except: return FALLBACK_TICKERS.copy()

# =============================================================================
# EXECUÇÃO PRINCIPAL
# =============================================================================

logger.iniciar_etapa("Coleta de Tickers")
tickers_b3 = obter_tickers_b3()
logger.concluir_etapa("Coleta de Tickers", {'total': len(tickers_b3)})
logger.log(f"✅ {len(tickers_b3)} tickers obtidos", "INFO")

tickers_yahoo = [t + ".SA" for t in tickers_b3]
tickers_liquidos = []
BATCH = 50

logger.iniciar_etapa("Filtro de Liquidez")
for i in range(0, len(tickers_yahoo), BATCH):
    batch = tickers_yahoo[i:i+BATCH]
    try:
        data = yf.download(batch, period='3mo', interval='1d', group_by='ticker', progress=False, auto_adjust=True)
        for t in batch:
            if t in TICKERS_BLOQUEADOS: continue
            try:
                if t not in data:
                    if LOG_FILTROS_DETALHADO:
                        logger.log(f"Ticker não encontrado no Yahoo: {t}", "DEBUG")
                    log_evento('TICKER_NAO_ENCONTRADO', t, {'motivo': 'delisted ou sem dados'})
                    continue
                df = data[t].copy()
                if isinstance(df.columns, pd.MultiIndex): df.columns = df.columns.droplevel(1)
                df.columns = [c.lower() for c in df.columns]
                df.rename(columns={'close':'Close','high':'High','low':'Low','open':'Open','volume':'Volume'}, inplace=True)
                if 'Volume' not in df.columns or df.empty:
                    continue
                vol = df['Volume'].rolling(21).mean().iloc[-1]
                px = df['Close'].iloc[-1]
                if pd.notna(vol) and pd.notna(px) and vol >= VOLUME_MINIMO_ACAO and (vol * px) >= VOLUME_FINANCEIRO_MINIMO:
                    tickers_liquidos.append(t)
            except Exception as e:
                if LOG_FILTROS_DETALHADO:
                    logger.log(f"Erro no filtro de liquidez: {t} - {str(e)[:50]}", "DEBUG")
                log_evento('ERRO_LIQUIDEZ', t, {'erro': str(e)})
                continue
    except Exception as e:
        logger.log(f"Erro no lote {i//BATCH}: {str(e)[:100]}", "ERRO")
    time.sleep(1)  # Rate limiting

if len(tickers_liquidos) < 10:
    logger.log("Poucos ativos líquidos, usando fallback", "WARN")
    tickers_liquidos = FALLBACK_TICKERS.copy()
logger.concluir_etapa("Filtro de Liquidez", {'liquidos': len(tickers_liquidos), 'total_inicial': len(tickers_yahoo)})
logger.log(f"💧 {len(tickers_liquidos)} ativos líquidos", "INFO")

# ✅ DOWNLOAD 5 ANOS COM LOGGING
logger.iniciar_etapa("Download de Dados (5 anos)")
data_d = {}
falhas = []
try:
    data_d_raw = yf.download(tickers_liquidos, period='5y', interval='1d', group_by='ticker', progress=False, auto_adjust=True)
    for t in tickers_liquidos:
        try:
            if t in data_d_raw:
                df_t = data_d_raw[t].copy()
                if isinstance(df_t.columns, pd.MultiIndex):
                    df_t.columns = df_t.columns.droplevel(1)
                df_t.columns = [col.lower() for col in df_t.columns]
                df_t.rename(columns={'close':'Close','high':'High','low':'Low','open':'Open','volume':'Volume'}, inplace=True)
                data_d[t] = df_t
            else:
                falhas.append(t)
        except:
            falhas.append(t)
    logger.log(f"Yahoo Finance: {len(data_d)} tickers baixados", "INFO")
except Exception as e:
    logger.log(f"Yahoo Finance falhou para download em lote: {str(e)[:100]}", "ERRO")
    falhas = tickers_liquidos.copy()

if falhas:
    logger.log(f"⚠️ {len(falhas)} tickers falharam no Yahoo (sem fallback investpy nesta versão)", "WARN")

time.sleep(1)
logger.concluir_etapa("Download de Dados (5 anos)", {'sucesso': len(data_d), 'falhas': len(falhas)})

# Resample para semanal/mensal
def resample_tf(df: pd.DataFrame, freq: str, min_days: int = 4) -> pd.DataFrame:
    if df is None or df.empty: return None
    df = df.copy()
    if not isinstance(df.index, pd.DatetimeIndex): df.index = pd.to_datetime(df.index)
    agg = {'Open':'first', 'High':'max', 'Low':'min', 'Close':'last', 'Volume':'sum'}
    df_resampled = df.resample(freq, closed='left', label='left').agg(agg)
    if freq.startswith('W'):
        counts = df.resample(freq, closed='left', label='left').count()['Close']
        df_resampled = df_resampled[counts >= min_days]
    return df_resampled.dropna()

logger.iniciar_etapa("Resample para Semanal/Mensal")
data_w, data_m = {}, {}
for t in tickers_liquidos:
    try:
        if t in data_d and not data_d[t].empty:
            df_d = data_d[t].copy()
            if isinstance(df_d.columns, pd.MultiIndex): df_d.columns = df_d.columns.droplevel(1)
            data_w[t] = resample_tf(df_d, 'W-FRI')
            data_m[t] = resample_tf(df_d, 'ME')
    except: continue
logger.concluir_etapa("Resample para Semanal/Mensal", {'semanais': len(data_w), 'mensais': len(data_m)})
logger.log("✅ Dados gerados com sucesso", "INFO")

# Cache macro
cache_macro = {}
for t_ref, (bench, _) in MACRO_REFERENCE.items():
    if bench not in cache_macro:
        try:
            df_b = yf.download(bench, period='1y', interval='1wk', progress=False, auto_adjust=True)
            cache_macro[bench] = df_b['Close'].values if not df_b.empty else None
        except: cache_macro[bench] = None

# ✅ REGIME DE VOLATILIDADE COM LOGGING E FALLBACK ROBUSTO
logger.iniciar_etapa("Detecção de Regime de Volatilidade")
ibov = None

# Camada 1: ^IBOV via Yahoo
try:
    ibov_raw = yf.download("^IBOV", period='3mo', interval='1d', progress=False)
    if not ibov_raw.empty and 'Close' in ibov_raw.columns:
        ibov = ibov_raw['Close'].dropna()
        if len(ibov) >= 60:
            logger.log("✅ IBOV obtido via ^IBOV", "INFO")
except Exception as e:
    logger.log(f"⚠️ ^IBOV falhou: {str(e)[:80]}", "WARN")

# Camada 2: ^BVSP como alternativa
if ibov is None or len(ibov) < 60:
    try:
        bvsp_raw = yf.download("^BVSP", period='3mo', interval='1d', progress=False)
        if not bvsp_raw.empty and 'Close' in bvsp_raw.columns:
            ibov = bvsp_raw['Close'].dropna()
            if len(ibov) >= 60:
                logger.log("✅ IBOV obtido via ^BVSP", "INFO")
    except Exception as e:
        logger.log(f"⚠️ ^BVSP falhou: {str(e)[:80]}", "WARN")

# Camada 3: BOVA11.SA como proxy líquido
if ibov is None or len(ibov) < 60:
    try:
        bova_raw = yf.download("BOVA11.SA", period='3mo', interval='1d', progress=False)
        if not bova_raw.empty and 'Close' in bova_raw.columns:
            ibov = bova_raw['Close'].dropna()
            if len(ibov) >= 60:
                logger.log("✅ Regime detectado via BOVA11.SA (proxy)", "INFO")
    except Exception as e:
        logger.log(f"⚠️ BOVA11.SA falhou: {str(e)[:80]}", "WARN")

# Aplicar parâmetros conforme regime
if ibov is not None and len(ibov) >= 60:
    try:
        # ✅ CORREÇÃO: Passar index explicitamente para evitar erro pandas
        regime_vol = detectar_regime_volatilidade(pd.DataFrame({'Close': ibov.values}, index=ibov.index))
        PARAMS_ATIVOS.clear()
        PARAMS_ATIVOS.update(PARAMS_ALTA_VOL if regime_vol == 'ALTA' else PARAMS_BAIXA_VOL)
        logger.log(f"📊 Regime: {regime_vol} (Parâmetros atualizados)", "INFO")
    except Exception as e:
        logger.log(f"⚠️ Erro ao calcular regime: {str(e)[:80]}. Usando BAIXA VOL.", "WARN")
        regime_vol = 'BAIXA'
        PARAMS_ATIVOS.clear()
        PARAMS_ATIVOS.update(PARAMS_BAIXA_VOL)
else:
    logger.log("⚠️ Dados insuficientes para regime. Usando BAIXA VOL (conservador).", "WARN")
    regime_vol = 'BAIXA'
    PARAMS_ATIVOS.clear()
    PARAMS_ATIVOS.update(PARAMS_BAIXA_VOL)

logger.concluir_etapa("Detecção de Regime de Volatilidade", {'regime': regime_vol})

kelly_pct = fractional_kelly(WIN_RATE_ESTIMADO, PAYOFF_ESTIMADO, PARAMS_ATIVOS['kelly_frac'])
risco_maximo = CAPITAL_TOTAL * kelly_pct

# Circuit breakers
def verificar_circuit_breakers() -> Tuple[bool, str]:
    if not os.path.exists(ARQUIVO_LOG): return True, None
    try:
        with open(ARQUIVO_LOG, 'r', encoding='utf-8') as f: logs = json.load(f)
        hoje = datetime.now().date()
        trades = [l for l in logs if l['tipo'] == 'TRADE_FECHADO' and datetime.fromisoformat(l['timestamp']).date() == hoje]
        if not trades: return True, None
        pnl = sum(t['dados'].get('pnl_real', 0) for t in trades)
        dd = abs(pnl) / CAPITAL_TOTAL
        if dd >= DRAWDOWN_MAX_DIARIO: return False, f"Drawdown {dd*100:.1f}% >= {DRAWDOWN_MAX_DIARIO*100}%"
        perdas = 0
        for t in reversed(trades):
            if t['dados'].get('pnl_real', 0) < 0: perdas += 1
            else: break
        if perdas >= MAX_PERDAS_CONSECUTIVAS: return False, f"{perdas} perdas seguidas (limite: {MAX_PERDAS_CONSECUTIVAS})"
        return True, None
    except: return True, None

pode, motivo = verificar_circuit_breakers()
if not pode:
    logger.log(f"🛑 CIRCUIT BREAKER ATIVADO: {motivo}", "ALERT")
    logger.log("⏸️ Execução pausada. Retorne amanhã.", "ALERT")
    raise SystemExit

oportunidades_swing = []
oportunidades_position = []

def get_df(data, ticker):
    # ✅ CORREÇÃO: Verificação explícita para evitar erro "truth value of DataFrame"
    if data is not None and isinstance(data, dict) and ticker in data:
        try:
            df = data[ticker].copy()
            if isinstance(df.columns, pd.MultiIndex):
                df.columns = df.columns.droplevel(1)
            df.rename(columns={'close':'Close','high':'High','low':'Low','open':'Open','volume':'Volume'}, inplace=True)
            return df
        except:
            pass
    return None

# Contadores para logging detalhado
stats_filtros = {
    'total_analisados': 0,
    'passou_preco': 0,
    'passou_risco': 0,
    'passou_confluencia': 0,
    'passou_mm': 0,
    'passou_macro': 0,
    'passou_wyckoff': 0,
    'passou_payoff': 0,
    'passou_gap': 0,
    'setup_aprovado': 0
}

logger.iniciar_etapa("Análise de Setups")

for i, ticker in enumerate(tickers_liquidos):
    if LOG_FILTROS_DETALHADO and i % 20 == 0:
        logger.log(f"Progresso: {i+1}/{len(tickers_liquidos)}", "DEBUG")
    
    df_w = get_df(data_w, ticker)
    df_d_local = get_df(data_d, ticker)
    df_m_local = get_df(data_m, ticker)
    
    if df_w is None or df_w.empty:
        continue
        
    stats_filtros['total_analisados'] += 1
    
    vol_fin = None
    try:
        vol_fin_calc = (df_w['Volume'] * df_w['Close']).rolling(20).mean()
        vol_fin = vol_fin_calc.iloc[-1] if pd.notna(vol_fin_calc.iloc[-1]) else None
    except:
        vol_fin = None
    
    # =====================================================================
    # SWING TRADE
    # =====================================================================
    res_swing = analisar_swing_trade(ticker, df_w=df_w, df_d=df_d_local)
    if res_swing:
        for r in res_swing:
            e = r['Entrada']
            rp = r['Risco (R$)'] / e
            
            # Filtro 1: Preço mínimo
            if e < PRECO_MINIMO: 
                if LOG_FILTROS_DETALHADO:
                    logger.log(f"❌ {ticker} Swing: Preço R$ {e:.2f} < mínimo R$ {PRECO_MINIMO}", "DEBUG")
                continue
            stats_filtros['passou_preco'] += 1
            
            # Filtro 2: Risco percentual
            if rp < RISCO_PERCENTUAL_MINIMO or rp > RISCO_PERCENTUAL_MAXIMO: 
                if LOG_FILTROS_DETALHADO:
                    logger.log(f"❌ {ticker} Swing: Risco {rp*100:.1f}% fora do range", "DEBUG")
                continue
            stats_filtros['passou_risco'] += 1
            
            # Filtro 3: Confluência (Regime + Eficiência)
            if EXIGIR_CONFLUENCIA:
                if r['Regime'] not in [1, 2] or r['Eficiência'] is None or r['Eficiência'] < 0.6: 
                    if LOG_FILTROS_DETALHADO:
                        logger.log(f"❌ {ticker} Swing: Falhou confluência (Regime={r['Regime']}, Ef={r['Eficiência']})", "DEBUG")
                    continue
            stats_filtros['passou_confluencia'] += 1
            
            # Filtro 4: Médias móveis
            mm200w = df_w['Close'].rolling(200).mean().iloc[-1]
            if pd.notna(mm200w):
                if r['Direcao'] == 'COMPRA' and e < mm200w: continue
                if r['Direcao'] == 'VENDA' and e > mm200w: continue
            if df_d_local is not None and not df_d_local.empty:
                mm200d = df_d_local['Close'].rolling(200).mean().iloc[-1]
                if pd.notna(mm200d):
                    if r['Direcao'] == 'COMPRA' and e < mm200d: continue
                    if r['Direcao'] == 'VENDA' and e > mm200d: continue
            stats_filtros['passou_mm'] += 1
            
            # Filtro 5: Volume anormal (se exigido)
            if PARAMS_ATIVOS.get('exigir_volume_anormal', False) and not r.get('Volume Anormal', True):
                if LOG_FILTROS_DETALHADO:
                    logger.log(f"❌ {ticker} Swing: Volume normal em alta vol", "DEBUG")
                log_evento('FILTRO_VOLUME', ticker, {'motivo': 'volume_normal_em_alta_vol'})
                continue
            
            # Filtro 6: Alinhamento macro
            alinhado_macro, _ = verificar_alinhamento_macro(ticker, r['Direcao'], cache_macro)
            if not alinhado_macro: 
                if LOG_FILTROS_DETALHADO:
                    logger.log(f"❌ {ticker} Swing: Falhou alinhamento macro", "DEBUG")
                continue
            stats_filtros['passou_macro'] += 1
            
            # Filtro 7: Wyckoff
            q_vol, s_vol = avaliar_qualidade_volume(df_w, r['Direcao'])
            wyck, w_conf = detectar_fase_wyckoff_adaptativo(df_w, r.get('Suporte'), r.get('Resistência'))
            if (r['Direcao'] == 'COMPRA' and wyck == 'DISTRIBUICAO') or (r['Direcao'] == 'VENDA' and wyck == 'ACUMULACAO'): 
                if LOG_FILTROS_DETALHADO:
                    logger.log(f"❌ {ticker} Swing: Wyckoff conflitante ({wyck})", "DEBUG")
                continue
            stats_filtros['passou_wyckoff'] += 1
            
            # Cálculo de score e sizing
            mult = calcular_score_confianca(r, alinhado_macro, q_vol, s_vol, wyck, w_conf)
            fat_liq = min(1.0, vol_fin / LIMITE_LIQUIDEZ_FINANCEIRA) if pd.notna(vol_fin) else 0.5
            
            lote_base = int(risco_maximo / r['Risco (R$)'])
            lote_aj = int(lote_base * mult * fat_liq)
            if lote_aj == 0: continue
            
            # Filtro 8: Payoff real
            al_rec, met_al = calcular_alvo_recomendado(r)
            p_real = calcular_payoff_real(e, al_rec, r['Stop Loss'], PARAMS_ATIVOS['custos_pct'])
            if p_real < 2.0: 
                if LOG_FILTROS_DETALHADO:
                    logger.log(f"❌ {ticker} Swing: Payoff real {p_real}:1 < 2.0:1", "DEBUG")
                continue
            stats_filtros['passou_payoff'] += 1
            
            # Filtro 9: Gap
            if len(df_w) >= 2:
                gap = abs(e - df_w['Close'].iloc[-2]) / df_w['Close'].iloc[-2]
                if gap > PARAMS_ATIVOS['gap_max_pct']:
                    if LOG_FILTROS_DETALHADO:
                        logger.log(f"❌ {ticker} Swing: Gap {gap*100:.1f}% > máximo {PARAMS_ATIVOS['gap_max_pct']*100:.1f}%", "DEBUG")
                    log_evento('GAP_FILTER', ticker, {'gap': gap})
                    continue
            stats_filtros['passou_gap'] += 1
            
            # Setup aprovado!
            r.update({
                'Alvo Recomendado': round(al_rec, 2),
                'Método Alvo': met_al,
                'Payoff Real': p_real,
                'Kelly %': round((kelly_pct * mult * fat_liq) * 100, 2),
                'Lote': lote_aj,
                'Score': int(mult * 100),
                'Fase Wyckoff': wyck,
                'Wyckoff Conf': round(w_conf, 2),
                'Regime Vol': regime_vol
            })
            
            if HABILITAR_LOGGING:
                log_evento('SETUP', ticker, {'mod': 'Swing', 'dir': r['Direcao'], 'entrada': e, 'stop': r['Stop Loss'], 'alvo': al_rec, 'score': r['Score']})
            
            stats_filtros['setup_aprovado'] += 1
            oportunidades_swing.append(r)
            if LOG_FILTROS_DETALHADO:
                logger.log(f"✅ {ticker} Swing APROVADO: Score={r['Score']}, Payoff={p_real}:1", "INFO")
    
    # =====================================================================
    # POSITION TRADE (lógica similar, omitida para brevidade - replicar filtros acima)
    # =====================================================================
    if df_m_local is not None and not df_m_local.empty:
        res_pos = analisar_position_trade(ticker, df_m=df_m_local, df_w=df_w)
        if res_pos:
            for r in res_pos:
                e = r['Entrada']
                rp = r['Risco (R$)'] / e
                
                # Aplicar mesmos filtros (simplificado para este exemplo)
                if e < PRECO_MINIMO or rp < 0.03 or rp > 0.30: continue
                if EXIGIR_CONFLUENCIA and (r['Regime'] not in [1, 2] or r['Eficiência'] is None or r['Eficiência'] < 0.5): continue
                
                mm50m = df_m_local['Close'].rolling(50).mean().iloc[-1]
                if pd.notna(mm50m) and ((r['Direcao'] == 'COMPRA' and e < mm50m) or (r['Direcao'] == 'VENDA' and e > mm50m)): continue
                
                alinhado_macro, _ = verificar_alinhamento_macro(ticker, r['Direcao'], cache_macro)
                if not alinhado_macro: continue
                
                q_vol, s_vol = avaliar_qualidade_volume(df_m_local, r['Direcao'])
                wyck, w_conf = detectar_fase_wyckoff_adaptativo(df_m_local, r.get('Suporte'), r.get('Resistência'))
                if (r['Direcao'] == 'COMPRA' and wyck == 'DISTRIBUICAO') or (r['Direcao'] == 'VENDA' and wyck == 'ACUMULACAO'): continue
                
                mult = calcular_score_confianca(r, alinhado_macro, q_vol, s_vol, wyck, w_conf)
                vol_fin_m = None
                try:
                    vol_fin_m_calc = (df_m_local['Volume'] * df_m_local['Close']).rolling(20).mean()
                    vol_fin_m = vol_fin_m_calc.iloc[-1] if pd.notna(vol_fin_m_calc.iloc[-1]) else None
                except:
                    vol_fin_m = None
                fat_liq_m = min(1.0, vol_fin_m / LIMITE_LIQUIDEZ_FINANCEIRA) if pd.notna(vol_fin_m) else 0.5
                
                lote_base = int(risco_maximo / r['Risco (R$)'])
                lote_aj = int(lote_base * mult * fat_liq_m)
                if lote_aj == 0: continue
                
                al_rec, met_al = calcular_alvo_recomendado(r)
                p_real = calcular_payoff_real(e, al_rec, r['Stop Loss'], PARAMS_ATIVOS['custos_pct'])
                if p_real < 2.0: continue
                
                if len(df_m_local) >= 2:
                    gap = abs(e - df_m_local['Close'].iloc[-2]) / df_m_local['Close'].iloc[-2]
                    if gap > PARAMS_ATIVOS['gap_max_pct']:
                        log_evento('GAP_FILTER', ticker, {'gap': gap})
                        continue
                
                r.update({
                    'Alvo Recomendado': round(al_rec, 2),
                    'Método Alvo': met_al,
                    'Payoff Real': p_real,
                    'Kelly %': round((kelly_pct * mult * fat_liq_m) * 100, 2),
                    'Lote': lote_aj,
                    'Score': int(mult * 100),
                    'Fase Wyckoff': wyck,
                    'Wyckoff Conf': round(w_conf, 2),
                    'Regime Vol': regime_vol
                })
                
                if HABILITAR_LOGGING:
                    log_evento('SETUP', ticker, {'mod': 'Position', 'dir': r['Direcao'], 'entrada': e, 'stop': r['Stop Loss'], 'alvo': al_rec, 'score': r['Score']})
                
                oportunidades_position.append(r)
                if LOG_FILTROS_DETALHADO:
                    logger.log(f"✅ {ticker} Position APROVADO: Score={r['Score']}, Payoff={p_real}:1", "INFO")

logger.concluir_etapa("Análise de Setups", stats_filtros)

# Limitar setups por dia
if len(oportunidades_position) > MAX_SETUPS_POR_DIA:
    oportunidades_position = sorted(oportunidades_position, key=lambda x: x['Score'], reverse=True)[:MAX_SETUPS_POR_DIA]
    logger.log(f"⚠️ Limite de {MAX_SETUPS_POR_DIA} setups atingido. Top oportunidades selecionadas", "WARN")

# Output final
logger.log(f"\n🎯 Swing: {len(oportunidades_swing)} | Position: {len(oportunidades_position)} setups", "RESULTADO")
logger.log(f"   Kelly: {kelly_pct*100:.2f}% | Regime: {regime_vol}", "RESULTADO")

if oportunidades_swing: enviar_email_ou_exibir(oportunidades_swing, "Swing Trade", regime_vol)
if oportunidades_position: enviar_email_ou_exibir(oportunidades_position, "Position Trade", regime_vol)

# Resumo final com timing
logger.resumo_final()

logger.log("✅ Execução concluída. Sistema 4.7 - Logging Detalhado + Correções", "SUCCESS")
print("\n✅ Execução concluída. Sistema 4.7 - Logging Detalhado + Correções")
